# 05 — LightGBM + Bayesian hyperparameter optimisation (Member C: nisarg-ravi)
Milestones M4 (my model) and M6 (Bayesian HPO).

* Training data: candidate rows of **labelled** points of the 80 train trips (decision D6).
* CV: `GroupKFold(5)` by trip, trips 10 and 16 in the same group (D12), seed 42.
* Objective: mean grouped-CV **log loss**, because the HMM uses the probabilities as emissions, so they must be well calibrated. Per-point Top-1 is tracked alongside.
* Student-test is evaluated once per model and never used to choose anything.

Runtime on Colab CPU: about 40–60 min for 40 trials. Set `N_TRIALS` lower for a quick check.

In [ ]:
# ---- Config (edit if your paths differ) ----
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/paya5am/01_pathsnap.git"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP = "/content/drive/MyDrive/PathSnap/Student folder.zip"
    if not os.path.exists("/content/01_pathsnap"):
        subprocess.run(["git", "clone", REPO_URL, "/content/01_pathsnap"], check=True)
    os.chdir("/content/01_pathsnap/alt_lightgbm_pipeline")   # self-contained sub-project (own pathsnap)
    if not os.path.exists("data/train"):
        os.makedirs("data", exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", ZIP, "-d", "/content/raw"], check=True)
        subprocess.run("cp -r '/content/raw/Student folder/train' '/content/raw/Student folder/student_test' data/", shell=True, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "optuna", "folium"], check=True)
else:
    os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
sys.path.insert(0, os.getcwd())
DATA = "data"                       # contains train/ and student_test/
DATASET = "experiments/dataset"     # candidate tables
RESULTS = "experiments/results"     # all logged outputs of this notebook
FIGS = "reports/figures/c_models"
for d in [DATASET, RESULTS, FIGS]:
    os.makedirs(d, exist_ok=True)
print("working dir:", os.getcwd())

## 1. Candidate table
Builds candidates (rule D7), labels and features for all 90 trips. Labels/features come from `pathsnap/labels.py` and `features.py`. Those are stand-ins until A's and B's M3 versions are merged; re-run this notebook after that.

In [ ]:
import subprocess, sys
if not os.path.exists(f'{DATASET}/candidates_train.npz'):
    subprocess.run([sys.executable, 'scripts/build_dataset.py', DATA, DATASET], check=True)
import pandas as pd
edge = pd.read_csv(f'{DATASET}/trip_edge_cases.csv')
edge.groupby('partition')[['n_points','unlabelled_points']].sum()

## 2. Untuned LightGBM, Optuna TPE search, refit, student-test
All results are written to `experiments/results/` (JSON + CSV).

In [ ]:
N_TRIALS = 40
if not os.path.exists(f'{RESULTS}/lgbm_results.json'):
    subprocess.run([sys.executable, 'scripts/train_lgbm.py', DATASET, RESULTS, str(N_TRIALS)], check=True)
import json
R = json.load(open(f'{RESULTS}/lgbm_results.json'))
rows = []
for name in ['untuned', 'tuned']:
    r = R[name]; st = r['student_test']
    rows.append(dict(model='LightGBM '+name, cv_logloss=f"{r['cv_log_loss_mean']:.4f} ± {r['cv_log_loss_std']:.4f}",
                     cv_top1=f"{r['cv_top1_mean']:.4f} ± {r['cv_top1_std']:.4f}", cv_f1=round(r['cv_f1_mean'],4),
                     st_top1=round(st['top1'],4), st_logloss=round(st['log_loss'],4), st_precision=round(st['precision'],4),
                     st_recall=round(st['recall'],4), st_f1=round(st['f1'],4), st_miss_rate=round(st['miss_rate'],4),
                     fit_all_s=round(r['fit_all_s'],1), infer_us_per_row=round(st['inference_us_per_row'],2)))
comp = pd.DataFrame(rows); comp.to_csv(f'{RESULTS}/lgbm_tuned_vs_untuned.csv', index=False); comp

In [ ]:
print('Best params:', R['tuned']['params'])
print('HPO wall time: %.0f s, trials: %d, pruned: %d' % (R['tuned']['hpo_wall_s'], R['tuned']['n_trials'], R['tuned']['n_pruned']))
print('Student-test confusion matrix (tuned) [[TN, FP], [FN, TP]]:', R['tuned']['student_test']['confusion'])

## 3. Optimisation history and parameter effects

In [ ]:
import matplotlib.pyplot as plt, numpy as np
tr = pd.read_csv(f'{RESULTS}/optuna_trials.csv')
done = tr[tr.state == 'COMPLETE']
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(done.number, done.value, s=14, label='trial')
ax[0].plot(done.number, done.value.cummin(), color='C1', label='best so far')
ax[0].axhline(R['untuned']['cv_log_loss_mean'], ls='--', color='grey', label='untuned')
ax[0].set_xlabel('trial'); ax[0].set_ylabel('grouped-CV log loss'); ax[0].legend(); ax[0].set_title('Optuna TPE history')
ax[1].scatter(done.params_learning_rate, done.value, c=np.log(done.params_num_leaves), s=18)
ax[1].set_xscale('log'); ax[1].set_xlabel('learning_rate (colour = log num_leaves)'); ax[1].set_ylabel('CV log loss')
plt.tight_layout(); plt.savefig(f'{FIGS}/optuna_history.png', dpi=150); plt.show()

In [ ]:
imp = pd.read_csv(f'{RESULTS}/feature_importance.csv')
imp.plot.barh(x='feature', y='gain', figsize=(6, 5), legend=False).invert_yaxis()
plt.title('Tuned LightGBM feature importance (gain)'); plt.tight_layout(); plt.savefig(f'{FIGS}/lgbm_feature_importance.png', dpi=150); plt.show()

In [ ]:
folds = pd.DataFrame(R['tuned']['folds'])[['fold','log_loss','top1','miss_rate','f1','fit_s']]
folds  # stability across validation folds